# SinoNom OCR — Phase 2: Spatial Layout Analysis

**SinoNom Document Intelligence Project**  
**An Nam Nhất Thống Chí (HVH_004)**

---

This notebook handles **Phase 2** of the pipeline: running the Adaptive Horizontal Threshold (AHT) layout clustering algorithm to organize raw text bounding boxes (derived from OCR) into right-to-left vertical text columns.


## 0 · Environment Setup


In [1]:
# ─── Detect environment ────────────────────────────────────────────────────
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")
print(f"Python {sys.version}")

Running in Google Colab: False
Python 3.12.7 | packaged by conda-forge | (main, Oct  4 2024, 15:57:01) [Clang 17.0.6 ]


In [ ]:
# ─── Google Drive Mount & Project Path Discovery (Colab only) ────────────────
if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    repo_name = "SinoNomViet_Transliteration_OCR"
    repo_url = f"https://github.com/khang3004/{repo_name}.git"
    candidate_files = [
        "src/sinonom_ocr/data_scraper.py",
        "src/sinonom_ocr/spatial_layout_engine.py",
        "src/sinonom_ocr/alignment_validator.py",
    ]

    def has_project_files(path: Path) -> bool:
        return all((path / f).exists() for f in candidate_files)

    found_root = None

    # 1. Check if files are already in current directory or a subfolder of /content
    cwd = Path(os.getcwd())
    if has_project_files(cwd):
        found_root = cwd
    else:
        for p in Path("/content").glob("**/src/sinonom_ocr/data_scraper.py"):
            if has_project_files(p.parent.parent.parent):
                found_root = p.parent.parent.parent
                break

    # 2. If not found locally, try to find them on Google Drive
    if not found_root:
        drive_mounted = os.path.exists("/content/drive/MyDrive")
        if not drive_mounted:
            print("❓ Google Drive is not mounted. Mounting now...")
            try:
                from google.colab import drive

                drive.mount("/content/drive")
                drive_mounted = True
            except Exception as e:
                print(f"⚠️ Google Drive mount failed or skipped: {e}")

        if drive_mounted:
            drive_paths = [
                Path("/content/drive/MyDrive/SinoNomOCR/HVH_004"),
                Path(f"/content/drive/MyDrive/{repo_name}"),
                Path("/content/drive/MyDrive/SinoNomVietnamese_OCR_Project"),
                Path("/content/drive/MyDrive/SinoNomViet_Transliteration_OCR"),
            ]
            for dp in drive_paths:
                if dp.exists() and has_project_files(dp):
                    found_root = dp
                    print(f"🎯 Found project files in Drive: {found_root}")
                    break

            if not found_root:
                print("🔍 Searching Google Drive for project files...")
                for p in Path("/content/drive/MyDrive").glob("**/src/sinonom_ocr/data_scraper.py"):
                    if has_project_files(p.parent.parent.parent):
                        found_root = p.parent.parent.parent
                        print(f"🎯 Found project files in Drive subfolder: {found_root}")
                        break

    # 3. If still not found, clone the repository from GitHub
    if not found_root:
        print("📁 Project files not found. Cloning repository from GitHub...")
        try:
            subprocess.run(["git", "clone", repo_url, f"/content/{repo_name}"], check=True)
            found_root = Path(f"/content/{repo_name}")
            print("✅ Repository cloned successfully.")
        except Exception as e:
            print(f"❌ Failed to clone repository: {e}")

    if found_root:
        PROJECT_ROOT = str(found_root)
        os.chdir(PROJECT_ROOT)
        print(f"🎯 Project root set to: {PROJECT_ROOT}")
        print(f"🔄 Changed working directory to: {os.getcwd()}")
    else:
        PROJECT_ROOT = "/content"
        print("❌ ERROR: Could not locate project root.")
else:
    # Local Jupyter running in notebooks/ directory
    PROJECT_ROOT = os.path.abspath("..")
    os.chdir(PROJECT_ROOT)
    print(f"🎯 Local project root: {PROJECT_ROOT}")
    print(f"🔄 Changed working directory to: {os.getcwd()}")

In [ ]:
# ─── Install package & dependencies ────────────────────────────────────────
# Installs the package in editable mode, which resolves all dependencies from pyproject.toml
if IN_COLAB:
    # Install the package in editable mode to register sinonom_ocr package
    %pip install -q -e .
    print("✅ Package and dependencies installed on Colab.")
else:
    print("✅ Local environment is managed by uv (run 'make install' if needed).")

In [ ]:
# ─── Path configuration ────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path(PROJECT_ROOT)

# Add src/ directory to sys.path so we can import our sinonom_ocr package
SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Detect Google Drive paths (Local Mac or Colab)
LOCAL_DRIVE_PATH = Path("/Users/KhangDS/Library/CloudStorage/GoogleDrive-gausseuler159357@gmail.com/My Drive/SinoNomOCR/HVH_004")
COLAB_DRIVE_PATH = Path("/content/drive/MyDrive/SinoNomOCR/HVH_004")

# Determine active directories
if IN_COLAB and COLAB_DRIVE_PATH.exists():
    print("✨ Using Google Drive directories on Google Colab.")
    DATA_DIR = COLAB_DRIVE_PATH / "data"
    OUTPUT_DIR = COLAB_DRIVE_PATH / "output"
elif LOCAL_DRIVE_PATH.exists():
    print("✨ Using Google Drive directories on local macOS.")
    DATA_DIR = LOCAL_DRIVE_PATH / "data"
    OUTPUT_DIR = LOCAL_DRIVE_PATH / "output"
else:
    print("✨ Using local repository directories.")
    DATA_DIR = ROOT / "data"
    OUTPUT_DIR = ROOT / "output"

# Output directories
RAW_IMAGES_DIR = DATA_DIR / "raw_images"
OUTPUT_XML_DIR = OUTPUT_DIR / "xml"
OUTPUT_XLSX_DIR = OUTPUT_DIR / "excel"
DICT_DIR = DATA_DIR / "dicts"

for d in [RAW_IMAGES_DIR, OUTPUT_XML_DIR, OUTPUT_XLSX_DIR, DICT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy reference dictionary files if missing from target DICT_DIR
repo_dicts_dir = ROOT / "data" / "dicts"
if repo_dicts_dir.exists() and DICT_DIR != repo_dicts_dir:
    import shutil
    for dic_file in repo_dicts_dir.glob("*.dic"):
        target_file = DICT_DIR / dic_file.name
        if not target_file.exists():
            shutil.copy(dic_file, target_file)
            print(f"📋 Copied reference dictionary {dic_file.name} to {target_file}")

print("Directory structure initialized.")

---

## 1 · Import Layout Module


In [ ]:
import logging
from paddleocr import PaddleOCR
from sinonom_ocr.spatial_layout_engine import (
    BoundingBox,
    create_mock_multi_char_response,
    create_mock_ocr_response,
    process_page_layout,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(name)s | %(message)s",
)

print("✅ Spatial Layout Engine & PaddleOCR imported.")

---

## 4 · Mock OCR + Spatial Layout Processing

In production, replace `create_mock_pages()` with real OCR API calls.


In [ ]:
# ─── Real OCR Inference & Bounding Box Extraction ──────────────────────────

# Initialize PaddleOCR
print("Initializing PaddleOCR...")
ocr = PaddleOCR(lang='ch', device='gpu' if IN_COLAB else 'cpu')

# Get all sorted page image paths
image_paths = sorted(list(RAW_IMAGES_DIR.glob("page_*.jpg")))
print(f"Found {len(image_paths)} images in {RAW_IMAGES_DIR}")

# corpus_layout[chapter_idx][page_idx] = (columns, ordered_boxes)
corpus_layout: list[list[tuple]] = []
ch_layout = []

for idx, img_path in enumerate(image_paths, 1):
    print(f"Processing image {idx}/{len(image_paths)}: {img_path.name}...")
    try:
        result = ocr.predict(str(img_path))
        raw_boxes = []
        if result and result[0]:
            page_data = result[0]
            polys = page_data.get('dt_polys', [])
            texts = page_data.get('rec_texts', [])
            scores = page_data.get('rec_scores', [])
            
            for poly, text, score in zip(polys, texts, scores):
                polygon = [(float(pt[0]), float(pt[1])) for pt in poly]
                bbox = BoundingBox(raw_polygon=polygon, text=text, confidence=float(score))
                raw_boxes.append(bbox)
        
        if not raw_boxes:
            print(f"  ⚠️ No text detected on page {idx} - using placeholder.")
            bbox = BoundingBox.from_xyxy(50, 50, 100, 100, text="#", confidence=0.0)
            raw_boxes.append(bbox)
            
        # Run spatial layout engine on page boxes
        columns, ordered_boxes = process_page_layout(
            raw_boxes,
            alpha=0.5,
            min_boxes_per_column=1,
        )
        ch_layout.append((columns, ordered_boxes))
        print(f"  Success: sorted {len(raw_boxes)} boxes into {len(columns)} columns.")
    except Exception as e:
        print(f"  ❌ Error processing {img_path.name}: {e}")
        ch_layout.append(([], []))

corpus_layout.append(ch_layout)
print("\n✅ Real OCR & spatial layout processing complete.")

In [ ]:
print("✅ Spatial layout clustering performed during OCR step.")

---

## 5 · Export Layout Output for Phase 3

We serialize the resulting column structures and bounding boxes to `data/ocr_layout_output.json` so that they can be processed by Phase 3 (Text Alignment & Validation) without needing to re-run the OCR or layout detection step.


In [ ]:
# ─── Serialize layout to JSON ──────────────────────────────────────────────
import json

output_file = DATA_DIR / "ocr_layout_output.json"

serialized_corpus = []
for ch_idx, ch_layout in enumerate(corpus_layout, start=1):
    ch_data = []
    for pg_idx, (columns, ordered_boxes) in enumerate(ch_layout, start=1):
        pg_data = {
            "chapter": ch_idx,
            "page": pg_idx,
            "columns": [
                {
                    "column_id": col.column_id,
                    "x_center": col.x_center,
                    "x_span": col.x_span,
                    "boxes": [
                        {
                            "raw_polygon": box.raw_polygon,
                            "text": box.text,
                            "confidence": box.confidence,
                            "column_id": box.column_id,
                            "reading_idx": box.reading_idx,
                        }
                        for box in col.boxes
                    ],
                }
                for col in columns
            ],
            "ordered_boxes": [
                {
                    "raw_polygon": box.raw_polygon,
                    "text": box.text,
                    "confidence": box.confidence,
                    "column_id": box.column_id,
                    "reading_idx": box.reading_idx,
                }
                for box in ordered_boxes
            ],
        }
        ch_data.append(pg_data)
    serialized_corpus.append(ch_data)

with open(output_file, "w", encoding="utf-8") as fh:
    json.dump(serialized_corpus, fh, ensure_ascii=False, indent=2)

print(
    f"✅ Successfully exported spatial layout results of {len(serialized_corpus)} chapters to {output_file}"
)